# Regreening change — did places that greened between 2000 and 2025 cool?

Extends the park before/after study to the whole city. For ~25,000 points sampled across Bengaluru it exports,
for two dry-season periods (Jan–Apr **2000–2004** with Landsat 5/7 and **2022–2026** with Landsat 8/9):
- `ndvi_1`, `ndvi_2` — greenness
- `lst_1`, `lst_2` — land surface temperature (°C)
- `nbhd_1`, `nbhd_2` — mean LST of the surrounding 1 km (the point's own neighbourhood, as a local control)
- `mndwi_1`, `mndwi_2` — water index (to drop water points)
- `built_2000`, `built_2025` — built-up share (GHSL)

Output: `bengaluru_regreening_change.csv` in Google Drive. Put it in `data/`, then run
`python rq1_greening_change/change_analysis.py`.

# SETUP

In [ ]:
import ee
import geemap.core as geemap

In [ ]:
PROJECT_ID = 'still-algebra-425406-v5'

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project=PROJECT_ID)

In [ ]:
!pip install -q osmnx

## City boundary (same as the other notebooks, ~717 km²)

In [ ]:
import osmnx as ox
from shapely.geometry import mapping

city_gdf = ox.geocode_to_gdf('Bengaluru, Karnataka, India')
city = ee.Geometry(mapping(city_gdf.geometry.iloc[0].simplify(0.0005)))
print('City area (km2):', round(city.area(1).getInfo() / 1e6))

## Settings

In [ ]:
PERIOD_1 = (2000, 2004)   # Landsat 5 and 7
PERIOD_2 = (2022, 2026)   # Landsat 8 and 9
MONTHS = (1, 4)           # Jan-Apr, dry season
MAX_CLOUD = 60
N_POINTS = 25000
NBHD_RADIUS = 1000        # m, neighbourhood used as local control

## Landsat processing
Same cloud masking and scaling as `regreening_parks.ipynb`. Band names differ between Landsat 5/7 and 8/9.

In [ ]:
BANDS_57 = dict(green='SR_B2', red='SR_B3', nir='SR_B4', swir1='SR_B5', st='ST_B6')
BANDS_89 = dict(green='SR_B3', red='SR_B4', nir='SR_B5', swir1='SR_B6', st='ST_B10')

def prep(img, b):
    qa = img.select('QA_PIXEL')
    clear = (qa.bitwiseAnd(1 << 1).eq(0)
             .And(qa.bitwiseAnd(1 << 3).eq(0))
             .And(qa.bitwiseAnd(1 << 4).eq(0)))
    sr = lambda name: img.select(b[name]).multiply(0.0000275).add(-0.2)
    lst = img.select(b['st']).multiply(0.00341802).add(149.0).subtract(273.15).rename('lst')
    ndvi = sr('nir').subtract(sr('red')).divide(sr('nir').add(sr('red'))).rename('ndvi')
    mndwi = sr('green').subtract(sr('swir1')).divide(sr('green').add(sr('swir1'))).rename('mndwi')
    return lst.addBands([ndvi, mndwi]).updateMask(clear).updateMask(img.select(b['st']).gt(0))

def collection(ids, b, years):
    d0, d1 = f'{years[0]}-01-01', f'{years[1] + 1}-01-01'
    out = None
    for cid in ids:
        c = (ee.ImageCollection(cid).filterBounds(city).filterDate(d0, d1)
             .filter(ee.Filter.calendarRange(MONTHS[0], MONTHS[1], 'month'))
             .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))
             .map(lambda i: prep(i, b)))
        out = c if out is None else out.merge(c)
    return out

col_1 = collection(['LANDSAT/LT05/C02/T1_L2', 'LANDSAT/LE07/C02/T1_L2'], BANDS_57, PERIOD_1)
col_2 = collection(['LANDSAT/LC08/C02/T1_L2', 'LANDSAT/LC09/C02/T1_L2'], BANDS_89, PERIOD_2)
print('Scenes, period 1:', col_1.size().getInfo(), '| period 2:', col_2.size().getInfo())

comp_1 = col_1.median()
comp_2 = col_2.median()

## Neighbourhood temperature (mean LST within 1 km)

In [ ]:
def neighbourhood(lst):
    return (lst.reproject(crs='EPSG:32643', scale=90)
               .focalMean(radius=NBHD_RADIUS, kernelType='circle', units='meters'))

nbhd_1 = neighbourhood(comp_1.select('lst')).rename('nbhd_1')
nbhd_2 = neighbourhood(comp_2.select('lst')).rename('nbhd_2')

## Built-up share, 2000 and 2025 (GHSL)

In [ ]:
def built_share(year):
    b = ee.Image(f'JRC/GHSL/P2023A/GHS_BUILT_S/{year}').select('built_surface')
    return b.divide(ee.Image.pixelArea().reproject(b.projection())).rename(f'built_{year}')

stack = ee.Image.cat([
    comp_1.select(['ndvi', 'lst', 'mndwi'], ['ndvi_1', 'lst_1', 'mndwi_1']), nbhd_1,
    comp_2.select(['ndvi', 'lst', 'mndwi'], ['ndvi_2', 'lst_2', 'mndwi_2']), nbhd_2,
    built_share(2000), built_share(2025)]).toFloat()

## Quick look: change in greenness 2000–04 → 2022–26

In [ ]:
d_ndvi = comp_2.select('ndvi').subtract(comp_1.select('ndvi'))
Map = geemap.Map()
Map.centerObject(city, 11)
Map.addLayer(d_ndvi.clip(city), {'min': -0.3, 'max': 0.3, 'palette': ['8c510a', 'f6e8c3', '01665e']},
             'NDVI change (green = greened)')
Map.addLayer(comp_2.select('lst').clip(city),
             {'min': 28, 'max': 45, 'palette': ['040274', '30c8e2', '86e26e', 'ffff2b', 'ff9c2b', 'ff0000']},
             'LST 2022-26', False)
Map

## Sample points and export to Drive

In [ ]:
pts = stack.sample(region=city, scale=90, numPixels=N_POINTS, seed=7, geometries=True,
                   tileScale=8, dropNulls=True)
pts = pts.map(lambda f: f.set({'lon': f.geometry().coordinates().get(0),
                               'lat': f.geometry().coordinates().get(1)}))

task = ee.batch.Export.table.toDrive(
    collection=pts,
    description='bengaluru_regreening_change',
    fileFormat='CSV',
    selectors=['lon', 'lat', 'ndvi_1', 'lst_1', 'nbhd_1', 'mndwi_1',
               'ndvi_2', 'lst_2', 'nbhd_2', 'mndwi_2', 'built_2000', 'built_2025'])
task.start()
print('Export started. It will appear in your Google Drive as bengaluru_regreening_change.csv')

In [ ]:
# Re-run this cell to check progress (READY -> RUNNING -> COMPLETED)
print(task.status()['state'], task.status().get('error_message', ''))